# Chunking Strategies — How You Split Documents Matters

Three approaches compared:
1. **Fixed-size** — cut every N characters (simple, can break mid-sentence)
2. **Parent-child** — index small chunks, return full docs (precise search, full context)
3. **Semantic** — split at topic boundaries using embedding similarity

In [ ]:
sample_doc = """Taiwan Semiconductor Manufacturing Company (TSMC) is the world's largest 
dedicated independent semiconductor foundry. Headquartered in Hsinchu, Taiwan, it has a 
supplier reliability rating of 94.7% and produces over 50% of the world's outsourced chips. 
Key clients include Apple, NVIDIA, and Qualcomm. If TSMC experiences a disruption, the 
downstream impact includes Apple iPhone production halting within 4 weeks. NVIDIA GPU supply 
drops by 60%, and automotive chip shortages cascade to 15+ OEMs."""

print(f"Full document: {len(sample_doc)} characters")
print(sample_doc)

## Strategy 1: Fixed-Size Chunking

In [ ]:
def fixed_chunk(text, size=150, overlap=30):
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += size - overlap
    return chunks

fixed_chunks = fixed_chunk(sample_doc, size=150, overlap=30)
print(f"Fixed-size: {len(fixed_chunks)} chunks (150 chars, 30 overlap)\n")
for i, chunk in enumerate(fixed_chunks):
    print(f"Chunk {i}: \"{chunk}\"")
    print(f"         ({len(chunk)} chars)\n")

Notice how Chunk 1 starts mid-sentence! That's the problem with fixed-size.

## Strategy 2: Semantic Chunking

In [ ]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# Split into sentences
sentences = re.split(r'(?<=[.!?])\s+', sample_doc.strip())
embeddings = model.encode(sentences)

# Compute similarity between consecutive sentences
print("Consecutive sentence similarities:\n")
for i in range(len(embeddings) - 1):
    sim = np.dot(embeddings[i], embeddings[i+1]) / (
        np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i+1])
    )
    marker = " ← TOPIC BREAK" if sim < 0.5 else ""
    print(f"  S{i} ↔ S{i+1}: {sim:.4f}{marker}")
    print(f"    \"{sentences[i][:60]}...\"")
    print(f"    \"{sentences[i+1][:60]}...\"")
    print()

In [ ]:
# Build semantic chunks — split where similarity drops
threshold = 0.5
semantic_chunks = []
current = [sentences[0]]

for i in range(1, len(sentences)):
    sim = np.dot(embeddings[i-1], embeddings[i]) / (
        np.linalg.norm(embeddings[i-1]) * np.linalg.norm(embeddings[i])
    )
    if sim < threshold:
        semantic_chunks.append(" ".join(current))
        current = [sentences[i]]
    else:
        current.append(sentences[i])
if current:
    semantic_chunks.append(" ".join(current))

print(f"Semantic chunking: {len(semantic_chunks)} chunks (threshold={threshold})\n")
for i, chunk in enumerate(semantic_chunks):
    print(f"Chunk {i}: \"{chunk}\"\n")

## Key Takeaways

| Strategy | Pros | Cons |
|----------|------|------|
| Fixed-size | Simple, predictable | Breaks mid-sentence |
| Parent-child | Precise search + full context | Needs parent-child mapping |
| Semantic | Respects topic boundaries | Slower (needs embeddings), chunk sizes vary |

**Rule of thumb:** Start with fixed-size. If results lack context, try parent-child. If results mix unrelated topics, try semantic chunking.